In [3]:
#Imports
import pandas as pd
import numpy as np
from scipy.stats import f,t
from google.colab import files
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import Lasso

In [4]:
# Add Dataset
uploaded = files.upload()
df_housing = pd.read_csv('Revised - Merged Housing Dataset (by ZipCode).csv')
df_stab = pd.read_csv('final_zip_stabilization_dataset.csv')
# df_pluto_stab = pd.read_csv('pluto_with_rent_stabilization_flag.csv')
#Data Setup


Saving final_zip_stabilization_dataset.csv to final_zip_stabilization_dataset.csv
Saving Revised - Merged Housing Dataset (by ZipCode).csv to Revised - Merged Housing Dataset (by ZipCode).csv


In [6]:
# DATA PREP
df_housing['zip'] = pd.to_numeric(df_housing['zip'], errors='coerce')
df_housing = df_housing.dropna(subset=['zip'])
df_housing['zip'] = df_housing['zip'].astype(int)

#Merges data on shared components
df_merged = pd.merge(df_housing, df_stab, left_on='zip',right_on ='zipcode', how='inner')

# FEATURE ENGINEERING
# Calculate Housing Supply with Vacancy Rate
df_merged['vacancy_rate'] = df_merged['vacant_units'] / df_merged['housing_units_total']

# Homeownership Rate
df_merged['homeownership_rate'] = df_merged['owner_occupied_units'] / df_merged['housing_units_total']

# Construction Share (Everything built 2010 or later)
df_merged['new_construction_share'] = (df_merged['built_2020_or_later'] + df_merged['built_2010_2019']) / df_merged['housing_units_total']

# Renter Density (Share of units occupied by renters)
df_merged['renter_density'] = df_merged['renter_occupied_units'] / df_merged['housing_units_total']

# Pre-War Building Share (Built 1939 or earlier)
df_merged['pre_war_share'] = df_merged['built_1939_or_earlier'] / df_merged['housing_units_total']

df_merged.columns

Index(['Unnamed: 0', 'geoid', 'zip', 'borough',
       'rent_burden_total_renter_households', 'rent_lt_10pct_income',
       'rent_10_to_14_9pct_income', 'rent_15_to_19_9pct_income',
       'rent_20_to_24_9pct_income', 'rent_25_to_29_9pct_income',
       'rent_30_to_34_9pct_income', 'rent_35_to_39_9pct_income',
       'rent_40_to_49_9pct_income', 'rent_50pct_or_more_income',
       'rent_not_computed', 'housing_units_total', 'median_gross_rent',
       'median_household_income', 'occupancy_total_units', 'occupied_units',
       'vacant_units', 'tenure_total_occupied_units', 'owner_occupied_units',
       'renter_occupied_units', 'total_population', 'year_built_total_units',
       'built_2020_or_later', 'built_2010_2019', 'built_2000_2009',
       'built_1990_1999', 'built_1980_1989', 'built_1970_1979',
       'built_1960_1969', 'built_1950_1959', 'built_1940_1949',
       'built_1939_or_earlier', 'zipcode', 'total_units', 'stabilized_units',
       'stabilized_buildings', 'total_build

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso

# 1. VARIABLE SELECTION
feature_columns = [
    'rent_burden_total_renter_households', 'rent_lt_10pct_income',
    'rent_10_to_14_9pct_income', 'rent_15_to_19_9pct_income',
    'rent_20_to_24_9pct_income', 'rent_25_to_29_9pct_income',
    'rent_30_to_34_9pct_income', 'rent_35_to_39_9pct_income',
    'rent_40_to_49_9pct_income', 'rent_50pct_or_more_income',
    'rent_not_computed', 'housing_units_total', 'median_gross_rent',
    'median_household_income', 'occupancy_total_units', 'occupied_units',
    'vacant_units', 'tenure_total_occupied_units', 'owner_occupied_units',
    'renter_occupied_units', 'total_population', 'year_built_total_units',
    'built_2020_or_later', 'built_2010_2019', 'built_2000_2009',
    'built_1990_1999', 'built_1980_1989', 'built_1970_1979',
    'built_1960_1969', 'built_1950_1959', 'built_1940_1949',
    'built_1939_or_earlier', 'zipcode', 'total_units', 'stabilized_units',
    'stabilized_buildings', 'total_buildings', 'stabilization_share',
    'vacancy_rate', 'homeownership_rate', 'new_construction_share',
    'renter_density', 'pre_war_share'
]

# Clean the dataset
df_model = df_merged[feature_columns].replace([np.inf, -np.inf], np.nan).dropna()

Y = df_model['median_gross_rent'].values


Xk = df_model.drop(columns=['median_gross_rent'])

X_train, X_test, Y_train, Y_test = train_test_split(Xk.values, Y, test_size=0.20, random_state=42)

# 3. TRAIN LASSO MODEL
lasso_model = Lasso(alpha=1.0, random_state=42, max_iter=10000)
lasso_model.fit(X_train, Y_train)

predictions = lasso_model.predict(X_test)

# ======================================================================
# 4. FORMATTED LASSO OUTPUT
# ======================================================================
# Match the coefficients to their variable names
feature_names = Xk.columns.tolist()
coefficients = lasso_model.coef_

# Create a DataFrame for easy sorting
df_coef = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Absolute_Impact': np.abs(coefficients) # Used to rank importance regardless of +/-
})

# Split into categories
df_positive = df_coef[df_coef['Coefficient'] > 0].sort_values(by='Absolute_Impact', ascending=False)
df_negative = df_coef[df_coef['Coefficient'] < 0].sort_values(by='Absolute_Impact', ascending=False)
df_eliminated = df_coef[df_coef['Coefficient'] == 0]

print("======================================================================")
print("LASSO REGRESSION FEATURE IMPORTANCE (TARGET: MEDIAN GROSS RENT)")
print("======================================================================")

print("\n FEATURES DRIVING RENT UP (Positive Impact)")
print("-" * 65)
if not df_positive.empty:
    for index, row in df_positive.iterrows():
        print(f"{row['Feature']:<38} | +{row['Coefficient']:.6f}")
else:
    print("None")

print("\n FEATURES DRIVING RENT DOWN (Negative Impact)")
print("-" * 65)
if not df_negative.empty:
    for index, row in df_negative.iterrows():
        print(f"{row['Feature']:<38} | {row['Coefficient']:.6f}")
else:
    print("None")

print("\n  FEATURES ELIMINATED BY LASSO (Zero Impact / Redundant)")
print("-" * 65)
if not df_eliminated.empty:
    eliminated_list = df_eliminated['Feature'].tolist()
    print(", ".join(eliminated_list))
else:
    print("None")
print("\n======================================================================")

LASSO REGRESSION FEATURE IMPORTANCE (TARGET: MEDIAN GROSS RENT)

 FEATURES DRIVING RENT UP (Positive Impact)
-----------------------------------------------------------------
stabilized_buildings                   | +0.116274
built_2010_2019                        | +0.109214
rent_15_to_19_9pct_income              | +0.103995
built_1980_1989                        | +0.103271
rent_40_to_49_9pct_income              | +0.073778
rent_25_to_29_9pct_income              | +0.051766
built_1950_1959                        | +0.050323
zipcode                                | +0.037794
built_1970_1979                        | +0.034006
total_units                            | +0.026336
vacant_units                           | +0.022343
occupancy_total_units                  | +0.014357
median_household_income                | +0.011508
renter_occupied_units                  | +0.010862
total_population                       | +0.006383
housing_units_total                    | +0.001440

 FEATURE

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.848e+06, tolerance: 5.301e+03
  model = cd_fast.enet_coordinate_descent(
